![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# From Grids to Climate Insight: An Interactive North Atlantic Climate Notebook

A climate-data-first journey through real sea-surface temperature anomalies, regional comparison, and interactive map-based interpretation.

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the tools we need for a modern climate-data workflow. We will combine multidimensional data access, temporal aggregation, anomaly calculation, cartographic mapping, and interactive exploration.
</div>

Click the code cell below and press the **<code>▶</code> Run** button (or use `Shift + Enter`). The print messages will help you keep track of progress.

In [ ]:
print("Starting setup: importing multidimensional climate-analysis libraries...")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display

plt.style.use("seaborn-v0_8-white")

print("Success! Imports ready.")
print("We can now load a real climate data cube and begin exploring it.")

## 2. The Climate Question

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> The ocean stores enormous amounts of heat, so sea-surface temperature patterns are a powerful way to study climate variability and environmental change. Even in a compact notebook, we can learn a lot by combining maps, time series, and anomaly logic.
</div>

In this notebook, we will use a real gridded climate dataset to ask:

- How does sea-surface temperature vary across the North Atlantic?
- How do recent conditions compare with a baseline climatology?
- Do different regions within the same ocean basin experience climate variability in the same way?
- How sensitive are our conclusions to choices like month, year, and anomaly threshold?

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Our plan:</b> Load a public NetCDF-style dataset, inspect its metadata, subset a real study domain, calculate monthly anomalies relative to a baseline period, compare regions, and then explore the results interactively.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> After your first successful run, come back to the editable values and try a different warm-anomaly threshold or a different month in the explorer. Small analytical choices can change the story you tell from the same data.
</div>

## 3. Loading a Real Gridded Climate Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Open a compact but real climate dataset that is robust enough for classroom use and rich enough to support meaningful spatiotemporal analysis.
</div>

We will use the **NOAA Extended Reconstructed Sea Surface Temperature v5** example dataset, accessed through `xarray`. It is a real gridded product with time, latitude, and longitude dimensions.

In [ ]:
print("Loading the gridded climate dataset with xarray...")

# EDIT THIS VALUE: choose the tutorial dataset name
dataset_name = "ersstv5"

ds = xr.tutorial.open_dataset(dataset_name)
source_path = str(ds.encoding.get("source", ""))

if "sst" not in ds.data_vars:
    raise ValueError("Expected a variable called 'sst', but it was not found in the dataset.")

sst_global = ds["sst"].sortby("lat")

if float(sst_global.lon.max()) > 180:
    sst_global = sst_global.assign_coords(lon=(((sst_global.lon + 180) % 360) - 180)).sortby("lon")

available_years = pd.DatetimeIndex(sst_global["time"].values).year

print(f"Dataset loaded: {dataset_name}")
print(f"Time coverage detected: {available_years.min()} to {available_years.max()}")
print(f"Grid dimensions: {sst_global.sizes['lat']} latitudes × {sst_global.sizes['lon']} longitudes")
print("Dataset loaded successfully. We can now inspect its metadata and dimensional structure.")

## 4. Reading NetCDF Metadata and Thinking in Dimensions

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>What xarray is doing for us:</b> `xarray` keeps the data values attached to named dimensions and coordinates like <code>time</code>, <code>lat</code>, and <code>lon</code>. That makes slicing, grouping, and averaging much more readable than working with anonymous NumPy arrays.
</div>

Before we analyse anything, we should always inspect:
- dimensions
n- coordinates
- units
- missing values
- time coverage
- georeferencing conventions


In [ ]:
print("Inspecting metadata, dimensions, coordinates, and missing values...")

dataset_title = ds.attrs.get("title", "No title attribute found")
dataset_summary = ds.attrs.get("summary", "No summary attribute found")
time_units = "Unavailable"

try:
    if source_path and Path(source_path).exists():
        with nc.Dataset(source_path) as root:
            dataset_title = getattr(root, "title", dataset_title)
            dataset_summary = getattr(root, "summary", dataset_summary)
            if "time" in root.variables:
                time_units = getattr(root.variables["time"], "units", time_units)
except Exception as exc:
    dataset_summary = f"NetCDF4 metadata inspection fell back to xarray attributes. Details: {exc}"

missing_pct = float(sst_global.isnull().mean().item() * 100)

metadata_table = pd.DataFrame({
    "Property": [
        "Dataset title",
        "Primary variable",
        "Units",
        "Time coverage",
        "Coordinate system",
        "NetCDF time units",
        "Missing-value share"
    ],
    "Value": [
        dataset_title,
        "sst",
        sst_global.attrs.get("units", "Not provided"),
        f"{available_years.min()} to {available_years.max()}",
        "Latitude / longitude grid",
        time_units,
        f"{missing_pct:.2f}%"
    ]
})

dimension_table = pd.DataFrame({
    "Dimension": list(sst_global.sizes.keys()),
    "Length": [sst_global.sizes[key] for key in sst_global.sizes.keys()]
})

display(metadata_table)
display(dimension_table)

print("A quick xarray preview of the main DataArray:")
display(sst_global)

print("Coordinates inspected. We now know what the data cube contains and how it is organised.")

## 5. Subsetting Space and Time with xarray

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Focus on a practical study domain, choose a baseline period, and turn raw temperatures into a more interpretable climate indicator: monthly anomalies.
</div>

A raw temperature field is useful, but an **anomaly** is often more revealing because it compares each month with what is typical for that month in a reference period.

Here we will:
- subset the North Atlantic and European shelf waters
- define a baseline climatology
- calculate monthly sea-surface temperature anomalies
- count warm-anomaly months above a chosen threshold
- define two comparison regions for later analysis


In [ ]:
print("Selecting the study region and calculating climatology and anomalies...")

# EDIT THIS VALUE: spatial study domain for the North Atlantic climate story
analysis_lat_bounds = (0, 70)
analysis_lon_bounds = (-80, 20)

# EDIT THIS VALUE: first year used in the analysis subset
analysis_start_year = 1982

# EDIT THIS VALUE: baseline period for monthly climatology
baseline_start_year = 1982
baseline_end_year = 2011

# EDIT THIS VALUE: warm monthly anomaly threshold in degrees Celsius
warm_threshold_c = 0.8

sst = sst_global.sel(
    time=slice(f"{analysis_start_year}-01-01", None),
    lat=slice(analysis_lat_bounds[0], analysis_lat_bounds[1]),
    lon=slice(analysis_lon_bounds[0], analysis_lon_bounds[1])
)

latest_year = int(sst.time.dt.year.max().item())
baseline_end_year = min(baseline_end_year, latest_year - 1 if latest_year > baseline_start_year else latest_year)

baseline = sst.sel(time=slice(f"{baseline_start_year}-01-01", f"{baseline_end_year}-12-31"))
climatology = baseline.groupby("time.month").mean("time", skipna=True)
anomaly = sst.groupby("time.month") - climatology

annual_anomaly = anomaly.groupby("time.year").mean("time", skipna=True)
summer_anomaly = anomaly.where(anomaly.time.dt.month.isin([6, 7, 8]), drop=True).groupby("time.year").mean("time", skipna=True)
warm_month_count = (anomaly >= warm_threshold_c).groupby("time.year").sum("time")

regions = {
    "Scottish Shelf": {
        "lat": (55, 62),
        "lon": (-12, 2),
        "color": "#1f78b4"
    },
    "Subtropical Atlantic": {
        "lat": (25, 35),
        "lon": (-55, -35),
        "color": "#d95f02"
    }
}

def regional_mean(data_array, box):
    return data_array.sel(
        lat=slice(box["lat"][0], box["lat"][1]),
        lon=slice(box["lon"][0], box["lon"][1])
    ).mean(("lat", "lon"), skipna=True)

regional_anomaly_ts = {}
regional_climatology_ts = {}
regional_exceedance_ts = {}

for region_name, box in regions.items():
    regional_anomaly_ts[region_name] = regional_mean(anomaly, box)
    regional_climatology_ts[region_name] = regional_mean(climatology, box)
    regional_exceedance_ts[region_name] = (regional_anomaly_ts[region_name] >= warm_threshold_c).groupby("time.year").sum("time")

first_compare_year = int(summer_anomaly["year"].values[1]) if summer_anomaly.sizes["year"] > 1 else int(summer_anomaly["year"].values[0])
last_compare_year = int(summer_anomaly["year"].values[-1])

summary_table = pd.DataFrame({
    "Measure": [
        "Study domain",
        "Subset years",
        "Baseline period",
        "Variable",
        "Units",
        "Warm threshold"
    ],
    "Value": [
        f"{analysis_lon_bounds[0]} to {analysis_lon_bounds[1]}° lon, {analysis_lat_bounds[0]} to {analysis_lat_bounds[1]}° lat",
        f"{analysis_start_year} to {latest_year}",
        f"{baseline_start_year} to {baseline_end_year}",
        "Sea-surface temperature anomaly",
        sst.attrs.get("units", "Unknown"),
        f"{warm_threshold_c:.1f} °C above monthly climatology"
    ]
})

display(summary_table)
print("Time subset prepared. Monthly climatology, anomalies, and regional summaries are ready.")

## 6. Mapping Climate Patterns with Cartopy

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>What we are comparing:</b> The first two maps compare summer mean anomalies in an earlier and later year. The third map shows a threshold-based indicator: how many months in the later year exceeded the chosen warm-anomaly cut-off.
</div>

This gives us two different analytical lenses:
- a continuous anomaly surface
- a threshold exceedance count

That is a useful lesson in climate-data interpretation: the same data cube can support different but complementary summaries.

In [ ]:
print("Generating cartographic climate maps...")

proj = ccrs.PlateCarree()

def add_region_boxes(ax):
    for name, box in regions.items():
        rect = Rectangle(
            (box["lon"][0], box["lat"][0]),
            box["lon"][1] - box["lon"][0],
            box["lat"][1] - box["lat"][0],
            fill=False,
            edgecolor=box["color"],
            linewidth=2,
            transform=proj,
            zorder=5
        )
        ax.add_patch(rect)
        ax.text(
            box["lon"][0] + 1,
            box["lat"][0] + 1,
            name,
            transform=proj,
            fontsize=8,
            color=box["color"],
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5),
            zorder=6
        )

fig, axes = plt.subplots(1, 3, figsize=(19, 6), subplot_kw={"projection": proj})

map_specs = [
    (summer_anomaly.sel(year=first_compare_year), f"JJA SST anomaly\n{first_compare_year}", "RdBu_r", -2, 2),
    (summer_anomaly.sel(year=last_compare_year), f"JJA SST anomaly\n{last_compare_year}", "RdBu_r", -2, 2),
    (warm_month_count.sel(year=last_compare_year), f"Months above {warm_threshold_c:.1f} °C\nin {last_compare_year}", "YlOrRd", 0, 12)
]

image_handles = []

for ax, (field, title, cmap, vmin, vmax) in zip(axes, map_specs):
    ax.add_feature(cfeature.OCEAN, facecolor="#f7fbff", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="#f2efe9", zorder=0)
    img = field.plot(
        ax=ax,
        transform=proj,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False
    )
    image_handles.append(img)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_extent([analysis_lon_bounds[0], analysis_lon_bounds[1], analysis_lat_bounds[0], analysis_lat_bounds[1]], crs=proj)
    add_region_boxes(ax)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title(title, fontsize=12)

fig.colorbar(image_handles[0], ax=axes[:2], orientation="horizontal", fraction=0.05, pad=0.08, label="SST anomaly (°C)")
fig.colorbar(image_handles[2], ax=axes[2], orientation="horizontal", fraction=0.05, pad=0.08, label="Number of warm-anomaly months")
fig.suptitle("North Atlantic climate snapshots: anomaly comparison and threshold exceedance", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

print("Maps generated successfully.")

## 7. Regional Comparison with Plotly

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Link the maps to a temporal story. We will compare two regions and ask how their anomaly histories and seasonal cycles differ.
</div>

This is a useful reminder that maps alone are not enough. A region can look warm or cool in one month, but time-series context helps us see persistence, contrast, and variability.

In [ ]:
print("Preparing polished regional summaries with Plotly...")

annual_rows = []
climatology_rows = []
exceedance_rows = []

for region_name, region_series in regional_anomaly_ts.items():
    annual_series = region_series.groupby("time.year").mean("time", skipna=True)
    for year, value in zip(annual_series["year"].values, annual_series.values):
        annual_rows.append({
            "Region": region_name,
            "Year": int(year),
            "Annual mean anomaly (°C)": float(value)
        })

    climatology_series = regional_climatology_ts[region_name]
    for month, value in zip(climatology_series["month"].values, climatology_series.values):
        climatology_rows.append({
            "Region": region_name,
            "Month number": int(month),
            "Month": pd.Timestamp(2001, int(month), 1).strftime("%b"),
            "Monthly climatology (°C)": float(value)
        })

    exceed_series = regional_exceedance_ts[region_name]
    for year, value in zip(exceed_series["year"].values, exceed_series.values):
        exceedance_rows.append({
            "Region": region_name,
            "Year": int(year),
            f"Months above {warm_threshold_c:.1f} °C anomaly": int(value)
        })

annual_df = pd.DataFrame(annual_rows)
climatology_df = pd.DataFrame(climatology_rows)
exceedance_df = pd.DataFrame(exceedance_rows)

fig_annual = px.line(
    annual_df,
    x="Year",
    y="Annual mean anomaly (°C)",
    color="Region",
    markers=True,
    title="Regional annual mean SST anomalies relative to the baseline period",
    color_discrete_map={
        "Scottish Shelf": "#1f78b4",
        "Subtropical Atlantic": "#d95f02"
    }
)
fig_annual.update_layout(yaxis_title="Annual mean anomaly (°C)", xaxis_title="Year")

fig_clim = px.line(
    climatology_df,
    x="Month",
    y="Monthly climatology (°C)",
    color="Region",
    markers=True,
    title=f"Monthly climatology by region ({baseline_start_year}–{baseline_end_year})",
    category_orders={"Month": [pd.Timestamp(2001, m, 1).strftime("%b") for m in range(1, 13)]},
    color_discrete_map={
        "Scottish Shelf": "#1f78b4",
        "Subtropical Atlantic": "#d95f02"
    }
)
fig_clim.update_layout(yaxis_title="Mean SST (°C)", xaxis_title="Month")

fig_exceed = px.bar(
    exceedance_df,
    x="Year",
    y=f"Months above {warm_threshold_c:.1f} °C anomaly",
    color="Region",
    barmode="group",
    title="How often did each region experience unusually warm months?",
    color_discrete_map={
        "Scottish Shelf": "#1f78b4",
        "Subtropical Atlantic": "#d95f02"
    }
)
fig_exceed.update_layout(yaxis_title="Warm-anomaly months in each year", xaxis_title="Year")

fig_annual.show()
fig_clim.show()
fig_exceed.show()

print("Plotly summary ready.")

## 8. Interactive Explorer: Month, Year, and Threshold

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> This is where the multidimensional nature of climate data becomes tangible. You can change the month, year, and threshold to see how the map and regional summary respond.
</div>

Run the next cell, then experiment with the controls. This is a good way to feel how climate interpretation depends on both time and analytical choice.

In [ ]:
print("Building the interactive climate explorer...")

month_options = [(pd.Timestamp(2001, m, 1).strftime("%B"), m) for m in range(1, 13)]

year_widget = widgets.IntSlider(
    value=last_compare_year,
    min=int(anomaly.time.dt.year.min().item()),
    max=int(anomaly.time.dt.year.max().item()),
    step=1,
    description="Year:",
    continuous_update=False
)

month_widget = widgets.Dropdown(
    options=month_options,
    value=8,
    description="Month:"
)

threshold_widget = widgets.FloatSlider(
    value=warm_threshold_c,
    min=0.0,
    max=2.0,
    step=0.1,
    description="Warm threshold (°C):",
    continuous_update=False,
    readout_format=".1f",
    style={"description_width": "initial"}
)

def selected_month_field(year, month):
    subset = anomaly.where((anomaly.time.dt.year == year) & (anomaly.time.dt.month == month), drop=True)
    if subset.sizes.get("time", 0) == 0:
        raise ValueError("No monthly field was found for that selection.")
    return subset.isel(time=0)

def explore_climate(year, month, threshold):
    field = selected_month_field(year, month)
    month_name = pd.Timestamp(2001, month, 1).strftime("%B")

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=proj)
    ax.add_feature(cfeature.OCEAN, facecolor="#f7fbff", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="#f2efe9", zorder=0)
    img = field.plot(
        ax=ax,
        transform=proj,
        cmap="RdBu_r",
        vmin=-2,
        vmax=2,
        add_colorbar=False
    )
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_extent([analysis_lon_bounds[0], analysis_lon_bounds[1], analysis_lat_bounds[0], analysis_lat_bounds[1]], crs=proj)
    add_region_boxes(ax)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title(f"SST anomaly in {month_name} {year}", fontsize=13)
    cbar = plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("SST anomaly (°C)")
    plt.show()

    rows = []
    for region_name, box in regions.items():
        region_field = field.sel(
            lat=slice(box["lat"][0], box["lat"][1]),
            lon=slice(box["lon"][0], box["lon"][1])
        )
        rows.append({
            "Region": region_name,
            "Mean anomaly (°C)": float(region_field.mean(skipna=True).item()),
            f"Cells above {threshold:.1f} °C (%)": float((region_field >= threshold).mean(skipna=True).item() * 100)
        })

    rows.append({
        "Region": "Whole study domain",
        "Mean anomaly (°C)": float(field.mean(skipna=True).item()),
        f"Cells above {threshold:.1f} °C (%)": float((field >= threshold).mean(skipna=True).item() * 100)
    })

    summary_df = pd.DataFrame(rows).round(2)
    display(summary_df)

    bar_df = summary_df[summary_df["Region"] != "Whole study domain"].copy()
    fig_bar = px.bar(
        bar_df,
        x="Region",
        y="Mean anomaly (°C)",
        color="Region",
        text="Mean anomaly (°C)",
        title=f"Regional mean anomaly in {month_name} {year}",
        color_discrete_map={
            "Scottish Shelf": "#1f78b4",
            "Subtropical Atlantic": "#d95f02"
        }
    )
    fig_bar.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig_bar.update_layout(showlegend=False, xaxis_title="", yaxis_title="Mean anomaly (°C)")
    fig_bar.show()

display(widgets.HTML(
    "<div style='border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:12px; margin:8px 0; border-radius:4px;'>"
    "<b>Try this:</b> Compare the same month in an early and late year, then increase the threshold. Watch how the estimated share of unusually warm grid cells changes."
    "</div>"
))

controls = widgets.HBox([year_widget, month_widget, threshold_widget])
interactive_output = widgets.interactive_output(
    explore_climate,
    {"year": year_widget, "month": month_widget, "threshold": threshold_widget}
)

display(controls, interactive_output)
print("Interactive explorer ready.")

## 9. Responsible Interpretation and Limitations

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important limitations:</b> This notebook reveals meaningful patterns, but it does not prove climate causality or attribution. It is a compact exploratory workflow, not a full climate assessment.
</div>

Here are the main cautions to keep in mind:

- **Baseline choice matters.** A different reference period would change the anomaly values.
- **Monthly aggregation smooths short-term extremes.** Daily and sub-daily variability are not visible here.
- **Spatial resolution is coarse.** A grid cell represents a broad area, so coastal detail and sharp ocean fronts are simplified.
- **Reconstructed gridded products carry uncertainty.** They blend observations and methods, rather than measuring every point directly.
- **Thresholds are analytical choices.** A count of months above 0.8 °C is useful, but it is not a universal definition of a climate event.
- **Maps alone are not enough.** Spatial snapshots need to be read alongside time series and domain knowledge.
- **Short notebook workflows should stay cautious.** We can identify patterns and contrasts, but not make strong claims about long-term attribution from this exercise alone.

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Good scientific practice:</b> Use notebooks like this to build climate-data literacy. In a fuller project, you would compare datasets, test multiple baselines, examine uncertainty documentation, and connect the statistics to physical mechanisms and literature.
</div>

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have completed a real xarray-led climate workflow. You opened a NetCDF-style dataset, inspected metadata and dimensions, subset a meaningful study region, calculated anomalies relative to a baseline, compared regions, used cartopy for climate mapping, and explored the results interactively. That is exactly the kind of multidimensional reasoning used in modern climate and Earth-system analysis.
</div>

### Suggested next steps

- Try a different anomaly threshold and see how the exceedance story changes
- Compare winter and summer behaviour explicitly
- Test another study domain, such as the South Atlantic or a smaller shelf region
- Explore a second climate variable if a suitable gridded dataset is available
- Add multi-year smoothing or simple trend summaries for a more advanced extension

If you edit the analysis settings, re-run the notebook from top to bottom so the maps, summaries, and interactive explorer all stay consistent.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)